In [1]:
# Text Generation

<img src="https://i0.wp.com/amanxai.com/wp-content/uploads/2024/01/Building-a-Text-Generation-Model-using-Python.png?resize=1200%2C675&ssl=1"/>

In [11]:
#!pip install tensorflow-datasets

In [12]:
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
# Load the Tiny Shakespeare dataset


In [13]:
import tensorflow as tf
import numpy as np

# veri yükle
path_to_file = tf.keras.utils.get_file(
    'shakespeare.txt',
    'https://storage.googleapis.com/download.tensorflow.org/data/shakespeare.txt'
)

text = open(path_to_file, 'rb').read().decode(encoding='utf-8')

# karakterlere böl
vocab = sorted(set(text))

# modern tokenizer
ids_from_chars = tf.keras.layers.StringLookup(vocabulary=list(vocab), mask_token=None)
chars_from_ids = tf.keras.layers.StringLookup(
    vocabulary=ids_from_chars.get_vocabulary(),
    invert=True,
    mask_token=None
)

# encode
all_chars = tf.strings.unicode_split(text, 'UTF-8')
all_ids = ids_from_chars(all_chars)

In [14]:
seq_length = 100

dataset = tf.data.Dataset.from_tensor_slices(all_ids)

sequences = dataset.batch(seq_length + 1, drop_remainder=True)

def split_input_target(sequence):
    return sequence[:-1], sequence[1:]

dataset = sequences.map(split_input_target)

BATCH_SIZE = 64
BUFFER_SIZE = 10000

dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True)

In [20]:
vocab_size = len(ids_from_chars.get_vocabulary())
embedding_dim = 256
rnn_units = 1024

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, embedding_dim),
    tf.keras.layers.LSTM(rnn_units, return_sequences=True),
    tf.keras.layers.Dense(vocab_size)
])

In [21]:
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)

model.compile(optimizer='adam', loss=loss)

history = model.fit(dataset, epochs=50)

Epoch 1/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 454s 3s/step - loss: 2.6030
Epoch 2/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 444s 3s/step - loss: 1.9038
Epoch 3/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 444s 3s/step - loss: 1.6570
Epoch 4/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 442s 3s/step - loss: 1.5252
Epoch 5/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 439s 3s/step - loss: 1.4419
Epoch 6/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - loss: 1.3847
Epoch 7/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 439s 3s/step - loss: 1.3407
Epoch 8/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 438s 3s/step - loss: 1.3027
Epoch 9/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 440s 3s/step - loss: 1.2705
Epoch 10/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - loss: 1.2408
Epoch 11/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - loss: 1.2120
Epoch 12/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - loss: 1.1833
Epoch 13/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 444s 3s/step - loss: 1.1551
Epoch 14/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 441s 3s/step - loss: 0.9158
Epoch 21/50
172/172 ━━━━━━━━━━━━━━━━━━━━ 44

In [24]:
def generate_text(model, start_string, num_generate=400, temperature=0.8):
    input_chars = tf.strings.unicode_split(start_string, 'UTF-8')
    input_ids = ids_from_chars(input_chars).numpy()
    input_eval = tf.expand_dims(input_ids, 0)

    text_generated = []

    for i in range(num_generate):
        predictions = model(input_eval)
        
        predictions = predictions[:, -1, :] / temperature

        predicted_id = tf.random.categorical(predictions, 1)[0, 0].numpy()

        input_eval = tf.expand_dims([predicted_id], 0)

        next_char = chars_from_ids([predicted_id]).numpy()[0].decode('utf-8')
        text_generated.append(next_char)

    return start_string + ''.join(text_generated)

In [26]:
print(generate_text(model, start_string="ROMEO:", temperature=0.5))

ROMEO:
Than mane o mear nous he thiche t mer medearow me anongeameres al s t that o hinou me f tanearee inocoureshar wand t mf than menathal moreat fe hanor ard tho y t s INTha t mo me ou youno t, ar w te pr at thed,
Berinoure amer in he win are hand,
This sthas are t marine thicous tharind t, me my t malour thar meld ar anour oure are t ld t t y s s mow pre mound ite inound d owe t ase f my aranith my,


In [27]:
model.save("text_generator_model.keras")